# Notebook 4: Maximum Softmax Probability Baseline

This notebook evaluates Maximum Softmax Probability (MSP) as a simple baseline for detecting uncertainty and distinguishing in-distribution (ID) examples from out-of-distribution (OOD) inputs. MSP is the largest softmax probability produced by a classifier for an input, and is used here as a measure of the model's confidence in its predicted known class.

The workflow loads the saved logits from the previous notebook, converts them into softmax probabilities, and extracts the maximum probability for each ID and OOD example. These confidence scores are compared using AUROC and AUPR, while ROC analysis measures the OOD false-positive rate when approximately 95% of ID examples are retained.

The notebook also selects a confidence threshold using a fixed ID coverage target and calculates the proportion of OOD examples rejected below that threshold. This shows how many unfamiliar inputs could be rejected while preserving a chosen level of performance on familiar inputs, and highlights the trade-off between ID coverage and OOD rejection.

This is a confidence-threshold baseline rather than a learned abstention model. The notebook does not train a separate rejection mechanism; instead, it estimates the behavior of a possible decision rule in which high-confidence predictions are accepted and low-confidence predictions are rejected or treated as abstentions. The results provide a reference point for comparing the post-hoc methods in the following experiments.

## Import Libraries

In [25]:
from pathlib import Path
import torch
import pandas as pd
import numpy as np

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
)

## Load Saved Logits

In [26]:
LOGIT_DIR = Path("./logits")

mobile_id = torch.load(
    LOGIT_DIR / "mobilenet_id_seed42.pt",
    map_location="cpu",
)

mobile_grape = torch.load(
    LOGIT_DIR / "mobilenet_grape_seed42.pt",
    map_location="cpu",
)

mobile_corn = torch.load(
    LOGIT_DIR / "mobilenet_corn_seed42.pt",
    map_location="cpu",
)

efficient_id = torch.load(
    LOGIT_DIR / "efficientnet_id_seed42.pt",
    map_location="cpu",
)

efficient_grape = torch.load(
    LOGIT_DIR / "efficientnet_grape_seed42.pt",
    map_location="cpu",
)

efficient_corn = torch.load(
    LOGIT_DIR / "efficientnet_corn_seed42.pt",
    map_location="cpu",
)

## Implement MSP Baseline

Maximum Softmax Probability (MSP) - Hendrycks & Gimpel

In [27]:
def msp_score(logits):
    probs = torch.softmax(logits, dim=1)
    return probs.max(dim=1).values.numpy()

In [28]:
mobile_id_msp = msp_score(mobile_id["logits"])
mobile_grape_msp = msp_score(mobile_grape["logits"])
mobile_corn_msp = msp_score(mobile_corn["logits"])

efficient_id_msp = msp_score(efficient_id["logits"])
efficient_grape_msp = msp_score(efficient_grape["logits"])
efficient_corn_msp = msp_score(efficient_corn["logits"])

## Inspect the score distributions

In [29]:
def describe_scores(name, scores):
    print(name)
    print(pd.Series(scores).describe())
    print()

In [30]:
describe_scores("MobileNet ID", mobile_id_msp)
describe_scores("MobileNet Grape OOD", mobile_grape_msp)
describe_scores("MobileNet Corn OOD", mobile_corn_msp)

MobileNet ID
count    9159.000000
mean        0.992644
std         0.046746
min         0.183249
25%         0.999868
50%         0.999980
75%         0.999997
max         1.000000
dtype: float64

MobileNet Grape OOD
count    4062.000000
mean        0.782586
std         0.202026
min         0.195369
25%         0.619014
50%         0.843884
75%         0.969821
max         0.999991
dtype: float64

MobileNet Corn OOD
count    3852.000000
mean        0.700715
std         0.229663
min         0.152678
25%         0.510822
50%         0.714124
75%         0.925632
max         0.999990
dtype: float64



In [31]:
describe_scores("EfficientNet ID", efficient_id_msp)
describe_scores("EfficientNet Grape OOD", efficient_grape_msp)
describe_scores("EfficientNet Corn OOD", efficient_corn_msp)

EfficientNet ID
count    9159.000000
mean        0.996387
std         0.031395
min         0.088438
25%         0.999829
50%         0.999935
75%         0.999973
max         1.000000
dtype: float64

EfficientNet Grape OOD
count    4062.000000
mean        0.739006
std         0.225361
min         0.157471
25%         0.550985
50%         0.788355
75%         0.959005
max         0.999995
dtype: float64

EfficientNet Corn OOD
count    3852.000000
mean        0.683056
std         0.230350
min         0.049700
25%         0.500967
50%         0.705973
75%         0.896079
max         0.999985
dtype: float64



## OOD Evaluation Function

- *AUROC*:
It is approximately the probability that a randomly selected ID image receives a higher confidence score than a randomly selected OOD image.

- *Average Precision*:
It measures how accurately high-confidence predictions identify ID samples. A high value means that when the model assigns high confidence, those examples are usually ID examples.

Important!!!

These metrics do not mean that the model abstained from OOD predictions. They only show that the model’s confidence scores can separate ID and OOD examples.

In [2]:
def evaluate_ood(id_scores, ood_scores):
    y_true = np.concatenate([
        np.ones(len(id_scores)),
        np.zeros(len(ood_scores)),
    ])

    scores = np.concatenate([
        id_scores,
        ood_scores,
    ])

    auroc = roc_auc_score(
        y_true,
        scores,
    )

    aupr_id = average_precision_score(
        y_true,
        scores,
    )

    return {
        "auroc": auroc,
        "aupr_id": aupr_id,
    }

In [33]:
mobile_grape_metrics = evaluate_ood(
    mobile_id_msp,
    mobile_grape_msp,
)

mobile_corn_metrics = evaluate_ood(
    mobile_id_msp,
    mobile_corn_msp,
)

print("MobileNet Grape:")
print(mobile_grape_metrics)

print("MobileNet Corn:")
print(mobile_corn_metrics)

MobileNet Grape:
{'auroc': 0.970766365144174, 'aupr_id': 0.9882392959697653}
MobileNet Corn:
{'auroc': 0.979332771322648, 'aupr_id': 0.9919599626361315}


In [34]:
efficient_grape_metrics = evaluate_ood(
    efficient_id_msp,
    efficient_grape_msp,
)

efficient_corn_metrics = evaluate_ood(
    efficient_id_msp,
    efficient_corn_msp,
)

print("EfficientNet Grape:")
print(efficient_grape_metrics)

print("EfficientNet Corn:")
print(efficient_corn_metrics)

EfficientNet Grape:
{'auroc': 0.9809392482897876, 'aupr_id': 0.990733020076964}
EfficientNet Corn:
{'auroc': 0.9904727170852722, 'aupr_id': 0.9958310387332365}


In [35]:
mobile_combined_ood = np.concatenate([
    mobile_grape_msp,
    mobile_corn_msp,
])

efficient_combined_ood = np.concatenate([
    efficient_grape_msp,
    efficient_corn_msp,
])

mobile_combined_metrics = evaluate_ood(
    mobile_id_msp,
    mobile_combined_ood,
)

efficient_combined_metrics = evaluate_ood(
    efficient_id_msp,
    efficient_combined_ood,
)

print(mobile_combined_metrics)
print(efficient_combined_metrics)

{'auroc': 0.9749359123515888, 'aupr_id': 0.9821397297938131}
{'auroc': 0.9855794961796293, 'aupr_id': 0.9873244914777674}


## FPR95

If we want to retain roughly 95% of familiar inputs, how many unfamiliar inputs will we accidentally accept?

`fpr95`: the OOD acceptance rate at approximately 95% ID acceptance.

`threshold`: the confidence threshold used at that operating point.

`tpr`: the actual ID acceptance rate selected by the ROC curve.

In [36]:
def fpr_at_95_tpr(id_scores, ood_scores):
    y_true = np.concatenate([
        np.ones(len(id_scores)),
        np.zeros(len(ood_scores)),
    ])

    scores = np.concatenate([
        id_scores,
        ood_scores,
    ])

    fpr, tpr, thresholds = roc_curve(
        y_true,
        scores,
    )

    idx = np.argmin(
        np.abs(tpr - 0.95)
    )

    return {
        "fpr95": fpr[idx],
        "threshold": thresholds[idx],
        "tpr": tpr[idx],
    }

In [37]:
fpr_at_95_tpr(
    mobile_id_msp,
    mobile_combined_ood,
)

{'fpr95': np.float64(0.13659337882234016),
 'threshold': np.float64(0.9875280261039734),
 'tpr': np.float64(0.950103723113877)}

## Rejection rate at fixed ID Coverage

In [38]:
def threshold_for_id_coverage(id_scores, coverage=0.95):
    return np.quantile(
        id_scores,
        1 - coverage,
    )

In [39]:
def ood_rejection_rate(ood_scores, threshold):
    return np.mean(
        ood_scores < threshold
    )

In [40]:
threshold = threshold_for_id_coverage(
    mobile_id_msp,
    coverage=0.95,
)

grape_rejection = ood_rejection_rate(
    mobile_grape_msp,
    threshold,
)

corn_rejection = ood_rejection_rate(
    mobile_corn_msp,
    threshold,
)

print("Threshold:", threshold)
print("Grape rejection:", grape_rejection)
print("Corn rejection:", corn_rejection)

Threshold: 0.9875311
Grape rejection: 0.8335795174790743
Corn rejection: 0.8948598130841121


## Summary Results

In [41]:
results = []

In [42]:
def add_result(
    results,
    model_name,
    ood_name,
    id_scores,
    ood_scores,
):
    metrics = evaluate_ood(
        id_scores,
        ood_scores,
    )

    fpr95_result = fpr_at_95_tpr(
        id_scores,
        ood_scores,
    )

    threshold = threshold_for_id_coverage(
        id_scores,
        coverage=0.95,
    )

    rejection = ood_rejection_rate(
        ood_scores,
        threshold,
    )

    results.append({
        "model": model_name,
        "ood_dataset": ood_name,
        "auroc": metrics["auroc"],
        "aupr_id": metrics["aupr_id"],
        "fpr95": fpr95_result["fpr95"],
        "ood_rejection_at_95_id_coverage": rejection,
    })

In [43]:
add_result(
    results,
    "MobileNetV3-Small",
    "Grape",
    mobile_id_msp,
    mobile_grape_msp,
)

add_result(
    results,
    "MobileNetV3-Small",
    "Corn",
    mobile_id_msp,
    mobile_corn_msp,
)

add_result(
    results,
    "MobileNetV3-Small",
    "Combined",
    mobile_id_msp,
    mobile_combined_ood,
)

add_result(
    results,
    "EfficientNet-B0",
    "Grape",
    efficient_id_msp,
    efficient_grape_msp,
)

add_result(
    results,
    "EfficientNet-B0",
    "Corn",
    efficient_id_msp,
    efficient_corn_msp,
)

add_result(
    results,
    "EfficientNet-B0",
    "Combined",
    efficient_id_msp,
    efficient_combined_ood,
)

msp_results = pd.DataFrame(results)

msp_results

,model,ood_dataset,auroc,aupr_id,fpr95,ood_rejection_at_95_id_coverage
0,MobileNetV3-Small,Grape,0.970766,0.988239,0.166420,0.833580
1,MobileNetV3-Small,Corn,0.979333,0.991960,0.105140,0.894860
2,MobileNetV3-Small,Combined,0.974936,0.982140,0.136593,0.863407
3,EfficientNet-B0,Grape,0.980939,0.990733,0.064254,0.935746
4,EfficientNet-B0,Corn,0.990473,0.995831,0.018692,0.981308
5,EfficientNet-B0,Combined,0.985579,0.987324,0.042077,0.957923


In [44]:
RESULT_DIR = Path("./results")
RESULT_DIR.mkdir(exist_ok=True)

msp_results.to_csv(
    RESULT_DIR / "msp_baseline_seed42.csv",
    index=False,
)